# GA1-220501094-01-AA1-EV06 – Diagnóstico de calidad de datos
## Zona Fit Evolution: preparación, exploración y documentación

**Aprendiz:** Juan Diego Mejía Llano
**Programa:** Análisis y Desarrollo de Software (ADSO) – SENA Regional Antioquia, ficha 3223873
**Rol:** analista de datos responsable del diagnóstico inicial de calidad

Este cuaderno carga, inspecciona y diagnostica los ocho datasets del proyecto. **No limpia ni modifica los datos**: solo los describe, identifica problemas y propone acciones para la etapa de preparación.

# Momento 1. Reconocimiento del proyecto y de los datasets

## Proyecto
**Nombre:** Análisis de deserción de miembros en Zona Fit Evolution.

**Propósito:** identificar qué hace que un miembro del gimnasio deje de renovar su membresía, para que el administrador pueda actuar a tiempo con los miembros en riesgo.

## Pregunta analítica (EV02)
> ¿Qué factores (tipo de plan, frecuencia de asistencia, horario de entrenamiento y antigüedad) influyen en que un miembro de Zona Fit Evolution cancele o no renueve su membresía durante sus primeros seis meses?

## Datasets seleccionados (EV03)

| # | DataFrame | Archivo | Fuente | Qué contiene | Relación con la pregunta |
|---|---|---|---|---|---|
| 1 | `zf_planes` | `zonafit/planes.csv` | Base de datos de Zona Fit (simulada) | Catálogo de planes: tipo, duración, ingresos y precio | Factor **tipo de plan** |
| 2 | `zf_miembros` | `zonafit/miembros.csv` | Base de datos de Zona Fit (simulada) | Personas inscritas: género, nacimiento, comuna, canal, fecha de registro | Base para calcular la **antigüedad** y perfilar a los miembros |
| 3 | `zf_membresias` | `zonafit/membresias.csv` | Base de datos de Zona Fit (simulada) | Cada compra o renovación de un plan, con fechas y estado | De aquí sale la variable de **deserción** (renueva o no renueva) |
| 4 | `zf_pagos` | `zonafit/pagos.csv` | Base de datos de Zona Fit (simulada) | Pagos de cada membresía: valor, método y estado | Morosidad y valor del cliente |
| 5 | `zf_accesos` | `zonafit/accesos.csv` | Lector de huella de Zona Fit (simulado) | Cada intento de ingreso con fecha, hora y resultado | **Frecuencia de asistencia** y **horario de entrenamiento** |
| 6 | `kg_churn` | `kaggle/gym_churn_us.csv` | Kaggle (secundario) | 4.000 clientes de una cadena de gimnasios con la variable `Churn` | Referencia externa de qué variables se asocian a la deserción |
| 7 | `kg_actividad` | `kaggle/gym_attendance_cleaned_data.csv` | Kaggle (secundario) | 2.600 visitas con tipo de membresía, entrenamiento y asistencia | Referencia de patrones de asistencia y horario |
| 8 | `gov_escenarios` | `datos_gov/escenarios_deportivos_inder.csv` | datos.gov.co – INDER Medellín (gubernamental) | Escenarios deportivos públicos de Medellín con barrio y coordenadas | Contexto: competencia gratuita cerca de donde viven los miembros |

> Zona Fit Evolution todavía no tiene datos reales de operación. Las cinco tablas `zf_` son una exportación **simulada** de su base de datos MySQL (ver `data/raw/zonafit/DICCIONARIO.md`), con problemas típicos de un sistema en producción incluidos a propósito para practicar el diagnóstico.

## Aspectos de calidad a revisar
1. **Completitud:** valores faltantes en columnas clave (fechas, precios, género, comuna, método de pago).
2. **Tipos de datos:** fechas y montos que se carguen como texto en lugar de fecha o número.
3. **Consistencia de categorías:** una misma categoría escrita de varias formas (por ejemplo, `M` y `Masculino`).
4. **Unicidad:** miembros, pagos o accesos duplicados.
5. **Integridad entre tablas:** registros que apunten a miembros o membresías que no existen.
6. **Reglas del negocio:** estados de membresía que no correspondan a sus fechas, accesos permitidos sin membresía vigente, accesos fuera del horario de la sede (5:00 a 22:00).
7. **Validez:** valores imposibles (fechas de nacimiento de 1900, coordenadas fuera de Medellín, pagos en cero).

# Momento 2. Preparación del entorno y carga de datos

El entorno se preparó con un entorno virtual de Python (`.venv`) y las dependencias de `requirements.txt` (pandas 3.0.6 y JupyterLab). Los datasets están en `data/raw/`, organizados por fuente. Como el cuaderno está en `notebooks/`, las rutas empiezan con `../`.

Cada archivo se carga en un **DataFrame independiente**. El prefijo del nombre indica la fuente: `zf_` Zona Fit, `kg_` Kaggle y `gov_` datos.gov.co. Todos los archivos son CSV separados por coma y codificados en UTF-8, por eso `read_csv()` no necesita parámetros adicionales.

In [1]:
import warnings

import pandas as pd

# pandas 3 avisa que describe(include='object') cambiará en pandas 4.
# El comando sigue funcionando; se oculta el aviso para que la salida sea legible.
warnings.filterwarnings("ignore", category=pd.errors.Pandas4Warning)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

print("Versión de pandas:", pd.__version__)

Versión de pandas: 3.0.6


In [2]:
RUTA_ZONAFIT = "../data/raw/zonafit/"
RUTA_KAGGLE = "../data/raw/kaggle/"
RUTA_GOV = "../data/raw/datos_gov/"

# Zona Fit Evolution (fuente primaria, simulada)
zf_planes = pd.read_csv(RUTA_ZONAFIT + "planes.csv")
zf_miembros = pd.read_csv(RUTA_ZONAFIT + "miembros.csv")
zf_membresias = pd.read_csv(RUTA_ZONAFIT + "membresias.csv")
zf_pagos = pd.read_csv(RUTA_ZONAFIT + "pagos.csv")
zf_accesos = pd.read_csv(RUTA_ZONAFIT + "accesos.csv")

# Kaggle (fuente secundaria)
kg_churn = pd.read_csv(RUTA_KAGGLE + "gym_churn_us.csv")
kg_actividad = pd.read_csv(RUTA_KAGGLE + "gym_attendance_cleaned_data.csv")

# datos.gov.co (fuente gubernamental)
gov_escenarios = pd.read_csv(RUTA_GOV + "escenarios_deportivos_inder.csv")

### Verificación de la carga

In [3]:
datasets = {
    "zf_planes": zf_planes,
    "zf_miembros": zf_miembros,
    "zf_membresias": zf_membresias,
    "zf_pagos": zf_pagos,
    "zf_accesos": zf_accesos,
    "kg_churn": kg_churn,
    "kg_actividad": kg_actividad,
    "gov_escenarios": gov_escenarios,
}

for nombre, df in datasets.items():
    print(f"{nombre:<16} cargado: {df.shape[0]:>6} filas x {df.shape[1]:>2} columnas")

zf_planes        cargado:      6 filas x  6 columnas
zf_miembros      cargado:    908 filas x  7 columnas
zf_membresias    cargado:   3425 filas x  7 columnas
zf_pagos         cargado:   3572 filas x  6 columnas
zf_accesos       cargado:  57003 filas x  5 columnas
kg_churn         cargado:   4000 filas x 14 columnas
kg_actividad     cargado:   2600 filas x 10 columnas
gov_escenarios   cargado:   1071 filas x  5 columnas


Los ocho archivos se cargaron sin errores, cada uno en su propio DataFrame: cinco tablas de Zona Fit, dos datasets de Kaggle y uno de datos.gov.co. El más grande es `zf_accesos` (57.003 registros) y el más pequeño `zf_planes` (6 registros).

# Momento 3. Inspección de la estructura de los datasets

Para cada DataFrame se ejecutan los ocho comandos de la guía: `head()`, `tail()`, `shape`, `columns`, `dtypes`, `info()`, `describe()` y `describe(include='object')`. Después de cada uno se explican sus características principales.

> En pandas 3 las columnas de texto aparecen con el tipo `str` (en pandas 2 aparecían como `object`). `describe(include='object')` las sigue incluyendo.

## 3.1 zf_planes – catálogo de planes

In [4]:
print("head() – primeros registros")
display(zf_planes.head())

print("tail() – últimos registros")
display(zf_planes.tail())

print("shape:", zf_planes.shape)
print("columns:", list(zf_planes.columns))

head() – primeros registros


,id_plan,nombre_plan,tipo_plan,duracion_dias,ingresos_incluidos,precio
0,1,Mensual ilimitado,Ilimitado,30,NaN,90000
1,2,Trimestral ilimitado,Ilimitado,90,NaN,240000
2,3,Semestral ilimitado,Ilimitado,180,NaN,450000
3,4,Anual ilimitado,Ilimitado,365,NaN,840000
4,5,Tiquetera 12 ingresos,Tiquetera,30,12.0,60000


tail() – últimos registros


,id_plan,nombre_plan,tipo_plan,duracion_dias,ingresos_incluidos,precio
1,2,Trimestral ilimitado,Ilimitado,90,NaN,240000
2,3,Semestral ilimitado,Ilimitado,180,NaN,450000
3,4,Anual ilimitado,Ilimitado,365,NaN,840000
4,5,Tiquetera 12 ingresos,Tiquetera,30,12.0,60000
5,6,Mensual estudiante,Ilimitado,30,NaN,70000


shape: (6, 6)
columns: ['id_plan', 'nombre_plan', 'tipo_plan', 'duracion_dias', 'ingresos_incluidos', 'precio']


In [5]:
print("dtypes:")
print(zf_planes.dtypes)
print()
zf_planes.info()

dtypes:
id_plan                 int64
nombre_plan               str
tipo_plan                 str
duracion_dias           int64
ingresos_incluidos    float64
precio                  int64
dtype: object

<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_plan             6 non-null      int64  
 1   nombre_plan         6 non-null      str    
 2   tipo_plan           6 non-null      str    
 3   duracion_dias       6 non-null      int64  
 4   ingresos_incluidos  1 non-null      float64
 5   precio              6 non-null      int64  
dtypes: float64(1), int64(3), str(2)
memory usage: 420.0 bytes


In [6]:
print("describe() – variables numéricas")
display(zf_planes.describe())

print("describe(include='object') – variables de texto")
display(zf_planes.describe(include="object"))

describe() – variables numéricas


,id_plan,duracion_dias,ingresos_incluidos,precio
count,6.000000,6.000000,1.0,6.000000
mean,3.500000,120.833333,12.0,291666.666667
std,1.870829,133.282282,NaN,307207.855802
min,1.000000,30.000000,12.0,60000.000000
25%,2.250000,30.000000,12.0,75000.000000
50%,3.500000,60.000000,12.0,165000.000000
75%,4.750000,157.500000,12.0,397500.000000
max,6.000000,365.000000,12.0,840000.000000


describe(include='object') – variables de texto


,nombre_plan,tipo_plan
count,6,6
unique,6,2
top,Mensual ilimitado,Ilimitado
freq,1,5


**Características principales**

- **6 filas y 6 columnas**: un registro por plan.
- Tipos: 4 numéricas (`id_plan`, `duracion_dias`, `precio` como `int64` e `ingresos_incluidos` como `float64`) y 2 de texto (`nombre_plan`, `tipo_plan`).
- `tipo_plan` tiene 2 categorías: 5 planes `Ilimitado` y 1 `Tiquetera`.
- `ingresos_incluidos` solo tiene 1 valor no nulo (12, la tiquetera). Los 5 vacíos son **esperados**: los planes ilimitados no tienen límite de ingresos. Por tener vacíos, Pandas la cargó como `float64` aunque es un número entero.
- Los precios van de 60.000 a 840.000 COP y las duraciones de 30 a 365 días.

## 3.2 zf_miembros – personas inscritas

In [7]:
print("head() – primeros registros")
display(zf_miembros.head())

print("tail() – últimos registros")
display(zf_miembros.tail())

print("shape:", zf_miembros.shape)
print("columns:", list(zf_miembros.columns))

head() – primeros registros


,id_miembro,documento_hash,genero,fecha_nacimiento,comuna,canal_captacion,fecha_registro
0,1,DOC75516401,F,1994-03-15,Buenos Aires,Referido,2025-08-24
1,2,DOC51923647,M,1997-05-01,Belén,Instagram,2026-02-19
2,3,DOC47750342,F,2003-02-27,El Poblado,Instagram,2025-07-29
3,4,DOC33539774,F,1993-07-13,Belén,Walk-in,2026-08-02
4,5,DOC33058946,M,1968-11-14,Buenos Aires,Instagram,2026-05-29


tail() – últimos registros


,id_miembro,documento_hash,genero,fecha_nacimiento,comuna,canal_captacion,fecha_registro
903,904,DOC25082223,M,2001-09-21,Belén,Walk-in,2025-02-18
904,905,DOC92097750,F,2002-07-10,El Poblado,Convenio empresarial,2026-01-10
905,906,DOC14355577,M,1996-03-19,Castilla,Convenio empresarial,2026-06-08
906,907,DOC77775728,F,2004-11-27,NaN,Instagram,2025-04-08
907,908,DOC89472866,M,2010-07-15,Laureles-Estadio,Instagram,2026-03-17


shape: (908, 7)
columns: ['id_miembro', 'documento_hash', 'genero', 'fecha_nacimiento', 'comuna', 'canal_captacion', 'fecha_registro']


In [8]:
print("dtypes:")
print(zf_miembros.dtypes)
print()
zf_miembros.info()

dtypes:
id_miembro          int64
documento_hash        str
genero                str
fecha_nacimiento      str
comuna                str
canal_captacion       str
fecha_registro        str
dtype: object

<class 'pandas.DataFrame'>
RangeIndex: 908 entries, 0 to 907
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   id_miembro        908 non-null    int64
 1   documento_hash    908 non-null    str  
 2   genero            881 non-null    str  
 3   fecha_nacimiento  854 non-null    str  
 4   comuna            862 non-null    str  
 5   canal_captacion   908 non-null    str  
 6   fecha_registro    908 non-null    str  
dtypes: int64(1), str(6)
memory usage: 49.8 KB


In [9]:
print("describe() – variables numéricas")
display(zf_miembros.describe())

print("describe(include='object') – variables de texto")
display(zf_miembros.describe(include="object"))

describe() – variables numéricas


,id_miembro
count,908.00000
mean,454.50000
std,262.26132
min,1.00000
25%,227.75000
50%,454.50000
75%,681.25000
max,908.00000


describe(include='object') – variables de texto


,documento_hash,genero,fecha_nacimiento,comuna,canal_captacion,fecha_registro
count,908,881,854,862,908,908
unique,900,6,806,13,5,469
top,DOC16318631,M,1900-01-01,Laureles-Estadio,Instagram,2026-01-04
freq,2,446,9,163,300,10


**Características principales**

- **908 filas y 7 columnas**: un registro por persona inscrita.
- Solo `id_miembro` es numérica; las otras 6 columnas son texto, incluidas **dos fechas** (`fecha_nacimiento` y `fecha_registro`) que deberían ser de tipo fecha.
- `info()` muestra vacíos en `genero` (881 de 908), `fecha_nacimiento` (854) y `comuna` (862).
- `describe(include='object')` revela señales de alerta: `documento_hash` tiene **900 valores únicos en 908 filas** (hay documentos repetidos), `genero` tiene **6 categorías** cuando se esperan 2, `comuna` tiene 13, y la fecha de nacimiento más repetida es `1900-01-01` (9 veces), un valor de relleno.

## 3.3 zf_membresias – periodos de membresía

In [10]:
print("head() – primeros registros")
display(zf_membresias.head())

print("tail() – últimos registros")
display(zf_membresias.tail())

print("shape:", zf_membresias.shape)
print("columns:", list(zf_membresias.columns))

head() – primeros registros


,id_membresia,id_miembro,id_plan,fecha_inicio,fecha_fin,precio_pagado,estado
0,1,1,6,2025-08-24,2025-09-22,70000.0,Vencida
1,2,1,6,2026-02-14,2026-03-15,70000.0,Vencida
2,3,1,6,2026-03-16,2026-04-14,70000.0,Vencida
3,4,1,6,2026-04-24,2026-05-23,70000.0,Vencida
4,5,1,6,2026-05-24,2026-06-22,70000.0,Vencida


tail() – últimos registros


,id_membresia,id_miembro,id_plan,fecha_inicio,fecha_fin,precio_pagado,estado
3420,3421,899,1,2025-12-03,2026-01-01,90000.0,Vencida
3421,3422,899,6,2026-01-02,2026-01-31,70000.0,Vencida
3422,3423,900,6,2025-10-07,2025-11-05,70000.0,Vencida
3423,3424,900,6,2025-11-06,2025-12-05,70000.0,Vencida
3424,3425,900,6,2025-12-06,2026-01-04,70000.0,Vencida


shape: (3425, 7)
columns: ['id_membresia', 'id_miembro', 'id_plan', 'fecha_inicio', 'fecha_fin', 'precio_pagado', 'estado']


In [11]:
print("dtypes:")
print(zf_membresias.dtypes)
print()
zf_membresias.info()

dtypes:
id_membresia       int64
id_miembro         int64
id_plan            int64
fecha_inicio         str
fecha_fin            str
precio_pagado    float64
estado               str
dtype: object

<class 'pandas.DataFrame'>
RangeIndex: 3425 entries, 0 to 3424
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id_membresia   3425 non-null   int64  
 1   id_miembro     3425 non-null   int64  
 2   id_plan        3425 non-null   int64  
 3   fecha_inicio   3425 non-null   str    
 4   fecha_fin      3425 non-null   str    
 5   precio_pagado  3374 non-null   float64
 6   estado         3425 non-null   str    
dtypes: float64(1), int64(3), str(3)
memory usage: 187.4 KB


In [12]:
print("describe() – variables numéricas")
display(zf_membresias.describe())

print("describe(include='object') – variables de texto")
display(zf_membresias.describe(include="object"))

describe() – variables numéricas


,id_membresia,id_miembro,id_plan,precio_pagado
count,3425.000000,3425.000000,3425.000000,3374.000000
mean,1713.000000,454.992993,3.361752,117529.342027
std,988.856663,253.540041,2.263381,122341.403831
min,1.000000,1.000000,1.000000,54000.000000
25%,857.000000,240.000000,1.000000,70000.000000
50%,1713.000000,458.000000,3.000000,90000.000000
75%,2569.000000,669.000000,6.000000,90000.000000
max,3425.000000,900.000000,6.000000,840000.000000


describe(include='object') – variables de texto


,fecha_inicio,fecha_fin,estado
count,3425,3425,3425
unique,644,718,3
top,2026-01-04,2026-02-02,Vencida
freq,15,15,3027


**Características principales**

- **3.425 filas y 7 columnas**: cada fila es una compra o renovación de un plan.
- Tipos: `id_membresia`, `id_miembro` e `id_plan` son `int64`; `precio_pagado` es `float64`; `fecha_inicio`, `fecha_fin` y `estado` son texto (las fechas deberían ser tipo fecha).
- `precio_pagado` tiene 3.374 valores no nulos: faltan 51 precios. Va de 54.000 a 840.000 COP, coherente con el catálogo y sus descuentos.
- `estado` tiene 3 categorías; la más frecuente es `Vencida` (3.027 registros).
- Las membresías pertenecen a `id_miembro` del 1 al 900, aunque hay 908 miembros: algunos inscritos no tienen ninguna membresía.

## 3.4 zf_pagos – pagos de membresías

In [13]:
print("head() – primeros registros")
display(zf_pagos.head())

print("tail() – últimos registros")
display(zf_pagos.tail())

print("shape:", zf_pagos.shape)
print("columns:", list(zf_pagos.columns))

head() – primeros registros


,id_pago,id_membresia,fecha_pago,valor,metodo_pago,estado_pago
0,1,1,2025-08-24,70000,Efectivo,Aprobado
1,2,2,2026-02-14,70000,Efectivo,Aprobado
2,3,3,2026-03-16,70000,Transferencia,Aprobado
3,4,4,2026-04-24,70000,Transferencia,Aprobado
4,5,5,2026-05-24,70000,Transferencia,Pendiente


tail() – últimos registros


,id_pago,id_membresia,fecha_pago,valor,metodo_pago,estado_pago
3567,1766,1693,2026-06-14,81000,Transferencia,Aprobado
3568,1174,1120,2026-10-13,90000,Efectivo,Aprobado
3569,696,661,2026-08-11,240000,Efectivo,Aprobado
3570,1398,1332,2026-03-11,60000,Transferencia,Aprobado
3571,175,163,2026-09-27,90000,Efectivo,Aprobado


shape: (3572, 6)
columns: ['id_pago', 'id_membresia', 'fecha_pago', 'valor', 'metodo_pago', 'estado_pago']


In [14]:
print("dtypes:")
print(zf_pagos.dtypes)
print()
zf_pagos.info()

dtypes:
id_pago         int64
id_membresia    int64
fecha_pago        str
valor             str
metodo_pago       str
estado_pago       str
dtype: object

<class 'pandas.DataFrame'>
RangeIndex: 3572 entries, 0 to 3571
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   id_pago       3572 non-null   int64
 1   id_membresia  3572 non-null   int64
 2   fecha_pago    3572 non-null   str  
 3   valor         3572 non-null   str  
 4   metodo_pago   3464 non-null   str  
 5   estado_pago   3572 non-null   str  
dtypes: int64(2), str(4)
memory usage: 167.6 KB


In [15]:
print("describe() – variables numéricas")
display(zf_pagos.describe())

print("describe(include='object') – variables de texto")
display(zf_pagos.describe(include="object"))

describe() – variables numéricas


,id_pago,id_membresia
count,3572.000000,3572.000000
mean,1781.872340,1708.264278
std,1028.540021,991.056201
min,1.000000,1.000000
25%,890.750000,846.750000
50%,1781.500000,1707.500000
75%,2673.250000,2570.250000
max,3562.000000,3425.000000


describe(include='object') – variables de texto


,fecha_pago,valor,metodo_pago,estado_pago
count,3572,3572,3464,3572
unique,765,32,4,3
top,2026-01-04,90000,Efectivo,Aprobado
freq,16,1221,1058,3399


**Características principales**

- **3.572 filas y 6 columnas**.
- **`valor` se cargó como texto (`str`) y no como número.** Por eso `describe()` solo resume `id_pago` e `id_membresia`, y el valor del pago aparece en `describe(include='object')` con 32 valores únicos. Esto indica que hay montos escritos con otro formato.
- `metodo_pago` tiene 3.464 valores no nulos: faltan 108.
- `id_pago` llega hasta 3.562, pero la tabla tiene 3.572 filas: hay **identificadores repetidos**.
- `estado_pago` tiene 3 categorías; predomina `Aprobado` (3.399).

## 3.5 zf_accesos – registros del lector de huella

In [16]:
print("head() – primeros registros")
display(zf_accesos.head())

print("tail() – últimos registros")
display(zf_accesos.tail())

print("shape:", zf_accesos.shape)
print("columns:", list(zf_accesos.columns))

head() – primeros registros


,id_acceso,id_miembro,fecha_hora,resultado,dispositivo
0,1,623,2025-01-01 07:23:24,Permitido,LECTOR-01
1,2,279,2025-01-02 18:55:48,Permitido,LECTOR-01
2,3,478,2025-01-02 20:18:04,Permitido,LECTOR-01
3,4,478,2025-01-03 17:55:50,Permitido,LECTOR-01
4,5,148,2025-01-04 07:07:52,Permitido,LECTOR-01


tail() – últimos registros


,id_acceso,id_miembro,fecha_hora,resultado,dispositivo
56998,56999,636,2026-10-15 20:45:34,Permitido,LECTOR-01
56999,57000,295,2026-10-15 20:49:41,Permitido,LECTOR-01
57000,57001,554,2026-10-15 21:01:00,Permitido,LECTOR-01
57001,57002,495,2026-10-15 21:35:25,Permitido,LECTOR-01
57002,57003,270,2026-10-15 21:41:43,Permitido,LECTOR-01


shape: (57003, 5)
columns: ['id_acceso', 'id_miembro', 'fecha_hora', 'resultado', 'dispositivo']


In [17]:
print("dtypes:")
print(zf_accesos.dtypes)
print()
zf_accesos.info()

dtypes:
id_acceso      int64
id_miembro     int64
fecha_hora       str
resultado        str
dispositivo      str
dtype: object



<class 'pandas.DataFrame'>
RangeIndex: 57003 entries, 0 to 57002
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   id_acceso    57003 non-null  int64
 1   id_miembro   57003 non-null  int64
 2   fecha_hora   57003 non-null  str  
 3   resultado    57003 non-null  str  
 4   dispositivo  55863 non-null  str  
dtypes: int64(2), str(3)
memory usage: 2.2 MB


In [18]:
print("describe() – variables numéricas")
display(zf_accesos.describe())

print("describe(include='object') – variables de texto")
display(zf_accesos.describe(include="object"))

describe() – variables numéricas


,id_acceso,id_miembro
count,57003.000000,57003.000000
mean,28502.000000,441.734979
std,16455.493034,252.040410
min,1.000000,1.000000
25%,14251.500000,219.000000
50%,28502.000000,439.000000
75%,42752.500000,657.000000
max,57003.000000,989.000000


describe(include='object') – variables de texto


,fecha_hora,resultado,dispositivo
count,57003,57003,55863
unique,56910,2,1
top,2025-02-06 16:04:18,Permitido,LECTOR-01
freq,2,56613,55863


**Características principales**

- **57.003 filas y 5 columnas**: es la tabla más grande del proyecto.
- `fecha_hora` está como texto y debe convertirse a fecha y hora para calcular frecuencia y franja horaria.
- `dispositivo` tiene 55.863 valores no nulos (faltan 1.140) y una sola categoría, `LECTOR-01`.
- `resultado` tiene 2 categorías: 56.613 `Permitido` y 390 `Denegado`.
- El `id_miembro` máximo es **989**, pero en `zf_miembros` solo existen 908 miembros: hay accesos de personas que no están registradas.

## 3.6 kg_churn – deserción de clientes de gimnasio (Kaggle)

In [19]:
print("head() – primeros registros")
display(kg_churn.head())

print("tail() – últimos registros")
display(kg_churn.tail())

print("shape:", kg_churn.shape)
print("columns:", list(kg_churn.columns))

head() – primeros registros


,gender,Near_Location,Partner,Promo_friends,Phone,Contract_period,Group_visits,Age,Avg_additional_charges_total,Month_to_end_contract,Lifetime,Avg_class_frequency_total,Avg_class_frequency_current_month,Churn
0,1,1,1,1,0,6,1,29,14.227470,5.0,3,0.020398,0.000000,0
1,0,1,0,0,1,12,1,31,113.202938,12.0,7,1.922936,1.910244,0
2,0,1,1,0,1,1,0,28,129.448479,1.0,2,1.859098,1.736502,0
3,0,1,1,1,1,12,1,33,62.669863,12.0,2,3.205633,3.357215,0
4,1,1,1,1,1,1,0,26,198.362265,1.0,3,1.113884,1.120078,0


tail() – últimos registros


,gender,Near_Location,Partner,Promo_friends,Phone,Contract_period,Group_visits,Age,Avg_additional_charges_total,Month_to_end_contract,Lifetime,Avg_class_frequency_total,Avg_class_frequency_current_month,Churn
3995,1,1,1,0,1,12,0,33,2.406023,12.0,8,2.284497,2.349070,0
3996,0,1,0,0,1,1,1,29,68.883764,1.0,1,1.277168,0.292859,1
3997,1,1,1,1,1,12,0,28,78.250542,11.0,2,2.786146,2.831439,0
3998,0,1,1,1,1,6,0,32,61.912657,5.0,3,1.630108,1.596237,0
3999,1,0,1,0,0,12,1,30,178.861814,12.0,1,1.101832,1.035270,0


shape: (4000, 14)
columns: ['gender', 'Near_Location', 'Partner', 'Promo_friends', 'Phone', 'Contract_period', 'Group_visits', 'Age', 'Avg_additional_charges_total', 'Month_to_end_contract', 'Lifetime', 'Avg_class_frequency_total', 'Avg_class_frequency_current_month', 'Churn']


In [20]:
print("dtypes:")
print(kg_churn.dtypes)
print()
kg_churn.info()

dtypes:
gender                                 int64
Near_Location                          int64
Partner                                int64
Promo_friends                          int64
Phone                                  int64
Contract_period                        int64
Group_visits                           int64
Age                                    int64
Avg_additional_charges_total         float64
Month_to_end_contract                float64
Lifetime                               int64
Avg_class_frequency_total            float64
Avg_class_frequency_current_month    float64
Churn                                  int64
dtype: object

<class 'pandas.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 14 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   gender                             4000 non-null   int64  
 1   Near_Location                      4000 non-null   int

In [21]:
print("describe() – variables numéricas")
display(kg_churn.describe())

print("describe(include='object') – variables de texto")
print("kg_churn no tiene columnas de texto (object/str): describe(include=\"object\") no aplica.")

describe() – variables numéricas


,gender,Near_Location,Partner,Promo_friends,Phone,Contract_period,Group_visits,Age,Avg_additional_charges_total,Month_to_end_contract,Lifetime,Avg_class_frequency_total,Avg_class_frequency_current_month,Churn
count,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000
mean,0.510250,0.845250,0.486750,0.308500,0.903500,4.681250,0.412250,29.184250,146.943728,4.322750,3.724750,1.879020,1.767052,0.265250
std,0.499957,0.361711,0.499887,0.461932,0.295313,4.549706,0.492301,3.258367,96.355602,4.191297,3.749267,0.972245,1.052906,0.441521
min,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,18.000000,0.148205,1.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,1.000000,0.000000,0.000000,1.000000,1.000000,0.000000,27.000000,68.868830,1.000000,1.000000,1.180875,0.963003,0.000000
50%,1.000000,1.000000,0.000000,0.000000,1.000000,1.000000,0.000000,29.000000,136.220159,1.000000,3.000000,1.832768,1.719574,0.000000
75%,1.000000,1.000000,1.000000,1.000000,1.000000,6.000000,1.000000,31.000000,210.949625,6.000000,5.000000,2.536078,2.510336,1.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,12.000000,1.000000,41.000000,552.590740,12.000000,31.000000,6.023668,6.146783,1.000000


describe(include='object') – variables de texto
kg_churn no tiene columnas de texto (object/str): describe(include="object") no aplica.


**Características principales**

- **4.000 filas y 14 columnas**, **todas numéricas** (10 `int64` y 4 `float64`) y **sin valores faltantes**. Como no tiene columnas de texto, `describe(include='object')` no aplica.
- Variables binarias codificadas como 0/1: `gender`, `Near_Location`, `Partner`, `Promo_friends`, `Phone`, `Group_visits` y `Churn`. El dataset no trae diccionario, así que no se sabe qué género corresponde a 0 y cuál a 1.
- `Churn` tiene media 0,265: el **26,5 % de los clientes desertó**. Es la variable objetivo de referencia.
- `Contract_period` solo toma los valores 1, 6 y 12 meses. `Month_to_end_contract` es `float64` aunque todos sus valores son enteros.
- La edad va de 18 a 41 años, un rango más estrecho que el de Zona Fit (15 años en adelante).

## 3.7 kg_actividad – visitas a gimnasio (Kaggle)

In [22]:
print("head() – primeros registros")
display(kg_actividad.head())

print("tail() – últimos registros")
display(kg_actividad.tail())

print("shape:", kg_actividad.shape)
print("columns:", list(kg_actividad.columns))

head() – primeros registros


,member_id,visit_date,age,gender,membership_type,workout_type,workout_duration_minutes,calories_burned,attendance_status,check_in_time
0,548,2024-07-01,18,Other,Annual,Strength Training,110,980,Absent,06:48:00
1,2578,2024-06-30,18,Male,Annual,CrossFit,77,417,Present,13:19:00
2,2100,2024-10-07,18,Other,Annual,Yoga,96,771,Absent,05:26:00
3,2264,2024-11-21,18,Female,Annual,Yoga,103,597,Absent,11:00:00
4,55,2024-05-07,18,Male,Annual,Cardio,53,483,Absent,15:03:00


tail() – últimos registros


,member_id,visit_date,age,gender,membership_type,workout_type,workout_duration_minutes,calories_burned,attendance_status,check_in_time
2595,1380,2024-05-12,65,Male,Quarterly,HIIT,63,375,Present,11:17:00
2596,2491,2024-03-14,65,Female,Quarterly,HIIT,70,603,Present,12:46:00
2597,505,2024-01-02,65,Male,Quarterly,HIIT,119,812,Absent,05:57:00
2598,2,2024-06-01,65,Female,Quarterly,Strength Training,72,650,Absent,19:17:00
2599,687,2024-02-06,65,Other,Quarterly,CrossFit,42,344,Absent,17:53:00


shape: (2600, 10)
columns: ['member_id', 'visit_date', 'age', 'gender', 'membership_type', 'workout_type', 'workout_duration_minutes', 'calories_burned', 'attendance_status', 'check_in_time']


In [23]:
print("dtypes:")
print(kg_actividad.dtypes)
print()
kg_actividad.info()

dtypes:
member_id                   int64
visit_date                    str
age                         int64
gender                        str
membership_type               str
workout_type                  str
workout_duration_minutes    int64
calories_burned             int64
attendance_status             str
check_in_time                 str
dtype: object

<class 'pandas.DataFrame'>
RangeIndex: 2600 entries, 0 to 2599
Data columns (total 10 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   member_id                 2600 non-null   int64
 1   visit_date                2600 non-null   str  
 2   age                       2600 non-null   int64
 3   gender                    2600 non-null   str  
 4   membership_type           2600 non-null   str  
 5   workout_type              2600 non-null   str  
 6   workout_duration_minutes  2600 non-null   int64
 7   calories_burned           2600 non-null   int64
 8   attend

In [24]:
print("describe() – variables numéricas")
display(kg_actividad.describe())

print("describe(include='object') – variables de texto")
display(kg_actividad.describe(include="object"))

describe() – variables numéricas


,member_id,age,workout_duration_minutes,calories_burned
count,2600.000000,2600.000000,2600.000000,2600.000000
mean,1300.500000,41.947308,70.162692,523.775385
std,750.699674,13.862067,28.610264,240.449717
min,1.000000,18.000000,20.000000,102.000000
25%,650.750000,30.000000,46.000000,329.000000
50%,1300.500000,42.000000,70.000000,504.000000
75%,1950.250000,54.000000,94.000000,691.250000
max,2600.000000,65.000000,120.000000,1186.000000


describe(include='object') – variables de texto


,visit_date,gender,membership_type,workout_type,attendance_status,check_in_time
count,2600,2600,2600,2600,2600,2600
unique,366,3,3,5,2,982
top,2024-12-02,Female,Annual,Strength Training,Absent,06:44:00
freq,16,886,903,559,1350,9


**Características principales**

- **2.600 filas y 10 columnas**, sin valores faltantes.
- `visit_date` y `check_in_time` están como texto; deben convertirse a fecha y hora.
- `member_id` tiene 2.600 valores distintos: cada miembro aparece **una sola vez**, así que no se puede seguir la asistencia de una persona en el tiempo.
- Las categorías están muy equilibradas: `gender` tiene 3 valores casi del mismo tamaño (`Female` 886, `Other` 863, `Male` 851), igual que `membership_type` (`Annual`, `Monthly`, `Quarterly`). Esa uniformidad sugiere un dataset generado, no observado.
- `attendance_status` tiene `Absent` (1.350) y `Present` (1.250). Las visitas son todas de 2024.

## 3.8 gov_escenarios – escenarios deportivos INDER Medellín (datos.gov.co)

In [25]:
print("head() – primeros registros")
display(gov_escenarios.head())

print("tail() – últimos registros")
display(gov_escenarios.tail())

print("shape:", gov_escenarios.shape)
print("columns:", list(gov_escenarios.columns))

head() – primeros registros


,nombre_escenario,direccion,barrio,latitud,longitud
0,Bolera Suramericana Unidad Deportiva de Belén ...,CARRERA 66 B 30 A 49,Rosales,6.23287,-75.58732
1,Campo de juego de tiro con arco Unidad Deporti...,CARRERA 66 B 31 15,Rosales,6.23404,-75.58735
2,Cancha Auxiliar de Hockey Guillermo León Boter...,CALLE 47 D 75 106,Estadio,6.25450,-75.59262
3,Cancha de baloncesto Aragón,CR 55D 53S,San Antonio de Prado,6.17593,-75.63822
4,Cancha de baloncesto Balcón de Miraflores,CALLE 42 22 B B,Cataluña,6.23331,-75.55018


tail() – últimos registros


,nombre_escenario,direccion,barrio,latitud,longitud
1066,Zona de lanzamiento de baloncesto N 2 Desarrol...,CALLE 63 84 131,Robledo,6.27377,-75.59647
1067,Zona de lanzamiento de baloncesto Unidad Depor...,CARRERA 69 101 25,Girardot,6.29717,-75.57113
1068,Zona de lanzamiento Estadio de Atletismo Alfon...,CALLE 48 B 72 60,Laureles,6.26018,-75.61270
1069,Zona de parkour Coliseo Carlos Mauro Hoyos,CALLE 5 65 B 2,Parque Juan Pablo II,6.21286,-75.58721
1070,Zona de salto Estadio de Atletismo Alfonso Galvis,CALLE 48 B 72 60,Estadio,6.26018,-75.61270


shape: (1071, 5)
columns: ['nombre_escenario', 'direccion', 'barrio', 'latitud', 'longitud']


In [26]:
print("dtypes:")
print(gov_escenarios.dtypes)
print()
gov_escenarios.info()

dtypes:
nombre_escenario        str
direccion               str
barrio                  str
latitud             float64
longitud            float64
dtype: object

<class 'pandas.DataFrame'>
RangeIndex: 1071 entries, 0 to 1070
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   nombre_escenario  1071 non-null   str    
 1   direccion         1071 non-null   str    
 2   barrio            1071 non-null   str    
 3   latitud           1071 non-null   float64
 4   longitud          1071 non-null   float64
dtypes: float64(2), str(3)
memory usage: 42.0 KB


In [27]:
print("describe() – variables numéricas")
display(gov_escenarios.describe())

print("describe(include='object') – variables de texto")
display(gov_escenarios.describe(include="object"))

describe() – variables numéricas


,latitud,longitud
count,1071.000000,1071.000000
mean,6.306759,-74.385911
std,2.188857,10.191078
min,1.234568,-75.702110
25%,6.233450,-75.598845
50%,6.253620,-75.581730
75%,6.283550,-75.559440
max,55.555555,55.555555


describe(include='object') – variables de texto


,nombre_escenario,direccion,barrio
count,1071,1071,1071
unique,1071,471,177
top,Bolera Suramericana Unidad Deportiva de Belén ...,CALLE 49 73 04,Estadio
freq,1,21,57


**Características principales**

- **1.071 filas y 5 columnas**, sin valores faltantes: 3 de texto (`nombre_escenario`, `direccion`, `barrio`) y 2 numéricas (`latitud`, `longitud`).
- `nombre_escenario` es único en todas las filas. `barrio` tiene 177 valores distintos.
- `describe()` muestra valores imposibles: la latitud de Medellín está cerca de 6,2, pero el máximo es **55,56** y el mínimo **1,23**; la longitud debería ser cercana a -75,6, pero el máximo es **55,56**.
- Las direcciones tienen espacios múltiples (por ejemplo, `CALLE 49    73    04`).
- El dataset trae **barrio** pero no **comuna**, mientras que `zf_miembros` trae comuna: no se pueden cruzar directamente.

## 3.9 Tabla de diagnóstico estructural

La tabla se construye con Pandas a partir de `shape` y `dtypes`, para que las cifras coincidan exactamente con los DataFrames.

In [28]:
observaciones = {
    "zf_planes": "Catálogo pequeño. Vacíos esperados en ingresos_incluidos (planes ilimitados).",
    "zf_miembros": "Fechas como texto; vacíos en género, nacimiento y comuna; categorías inconsistentes; documentos repetidos.",
    "zf_membresias": "Fechas como texto; 51 precios vacíos; estados que no coinciden con las fechas.",
    "zf_pagos": "Monto (valor) cargado como texto; IDs repetidos; método de pago vacío; fechas en dos formatos.",
    "zf_accesos": "Tabla más grande; fecha_hora como texto; dispositivo vacío; accesos de miembros inexistentes.",
    "kg_churn": "Todo numérico y completo; binarias 0/1 sin diccionario; sin fechas ni identificadores.",
    "kg_actividad": "Completo; fechas como texto; una visita por miembro; categorías muy uniformes.",
    "gov_escenarios": "Completo; coordenadas fuera de Medellín; direcciones con espacios; trae barrio, no comuna.",
}


def resumen_tipos(df):
    conteo = df.dtypes.astype(str).value_counts()
    return ", ".join(f"{tipo} ({n})" for tipo, n in conteo.items())


tabla_estructural = pd.DataFrame(
    [
        {
            "Dataset": nombre,
            "Filas": df.shape[0],
            "Columnas": df.shape[1],
            "Tipos de datos": resumen_tipos(df),
            "Observaciones": observaciones[nombre],
        }
        for nombre, df in datasets.items()
    ]
)

pd.set_option("display.max_colwidth", None)
display(tabla_estructural)

,Dataset,Filas,Columnas,Tipos de datos,Observaciones
0,zf_planes,6,6,"int64 (3), str (2), float64 (1)",Catálogo pequeño. Vacíos esperados en ingresos_incluidos (planes ilimitados).
1,zf_miembros,908,7,"str (6), int64 (1)","Fechas como texto; vacíos en género, nacimiento y comuna; categorías inconsistentes; documentos repetidos."
2,zf_membresias,3425,7,"int64 (3), str (3), float64 (1)",Fechas como texto; 51 precios vacíos; estados que no coinciden con las fechas.
3,zf_pagos,3572,6,"str (4), int64 (2)",Monto (valor) cargado como texto; IDs repetidos; método de pago vacío; fechas en dos formatos.
4,zf_accesos,57003,5,"str (3), int64 (2)",Tabla más grande; fecha_hora como texto; dispositivo vacío; accesos de miembros inexistentes.
5,kg_churn,4000,14,"int64 (10), float64 (4)",Todo numérico y completo; binarias 0/1 sin diccionario; sin fechas ni identificadores.
6,kg_actividad,2600,10,"str (6), int64 (4)",Completo; fechas como texto; una visita por miembro; categorías muy uniformes.
7,gov_escenarios,1071,5,"str (3), float64 (2)","Completo; coordenadas fuera de Medellín; direcciones con espacios; trae barrio, no comuna."


# Momento 4. Diagnóstico inicial de calidad

Se analizan los cinco aspectos de la guía: valores faltantes, tipos de datos, posibles inconsistencias, limitaciones y acciones de limpieza propuestas. Las comprobaciones **solo miden**; ningún dato se corrige ni se elimina.

La fecha de corte de la exportación de Zona Fit es el **15 de octubre de 2026** (ver `DICCIONARIO.md`). Se usa para comparar el estado de las membresías con sus fechas.

In [29]:
FECHA_CORTE = pd.Timestamp("2026-10-15")

## 4.1 Valores faltantes

In [30]:
filas = []
for nombre, df in datasets.items():
    faltantes = df.isna().sum()
    for columna, cantidad in faltantes[faltantes > 0].items():
        filas.append(
            {
                "Dataset": nombre,
                "Columna": columna,
                "Faltantes": cantidad,
                "% del total": round(cantidad / len(df) * 100, 2),
            }
        )

tabla_faltantes = pd.DataFrame(filas)
display(tabla_faltantes)

sin_faltantes = [n for n, df in datasets.items() if df.isna().sum().sum() == 0]
print("Datasets sin ningún valor faltante:", sin_faltantes)

,Dataset,Columna,Faltantes,% del total
0,zf_planes,ingresos_incluidos,5,83.33
1,zf_miembros,genero,27,2.97
2,zf_miembros,fecha_nacimiento,54,5.95
3,zf_miembros,comuna,46,5.07
4,zf_membresias,precio_pagado,51,1.49
5,zf_pagos,metodo_pago,108,3.02
6,zf_accesos,dispositivo,1140,2.00


Datasets sin ningún valor faltante: ['kg_churn', 'kg_actividad', 'gov_escenarios']


**Interpretación**

- **`zf_planes.ingresos_incluidos` (5 vacíos, 83,33 %)** no es un error: los planes ilimitados no tienen límite de ingresos. Se debe documentar, no imputar.
- **`zf_miembros`**: faltan `fecha_nacimiento` (54, 5,95 %), `comuna` (46, 5,07 %) y `genero` (27, 2,97 %). Sin fecha de nacimiento no se puede calcular la edad, y sin comuna no se puede relacionar al miembro con su entorno.
- **`zf_membresias.precio_pagado` (51, 1,49 %)**: afecta el cálculo de ingresos y del valor de cada cliente, pero no la deserción.
- **`zf_pagos.metodo_pago` (108, 3,02 %)**: afecta el análisis por medio de pago, no la pregunta central.
- **`zf_accesos.dispositivo` (1.140, 2 %)**: impacto bajo, porque la sede solo tiene un lector (`LECTOR-01`).
- Los datasets de Kaggle y de datos.gov.co **no tienen faltantes**. Ningún porcentaje supera el 6 % en columnas relevantes, así que los vacíos no impiden continuar.

## 4.2 Tipos de datos

In [31]:
columnas_fecha = {
    "zf_miembros": ["fecha_nacimiento", "fecha_registro"],
    "zf_membresias": ["fecha_inicio", "fecha_fin"],
    "zf_pagos": ["fecha_pago"],
    "zf_accesos": ["fecha_hora"],
    "kg_actividad": ["visit_date", "check_in_time"],
}

for nombre, columnas in columnas_fecha.items():
    for columna in columnas:
        print(f"{nombre}.{columna:<18} tipo actual: {datasets[nombre][columna].dtype}  (debería ser fecha/hora)")

zf_miembros.fecha_nacimiento   tipo actual: str  (debería ser fecha/hora)
zf_miembros.fecha_registro     tipo actual: str  (debería ser fecha/hora)
zf_membresias.fecha_inicio       tipo actual: str  (debería ser fecha/hora)
zf_membresias.fecha_fin          tipo actual: str  (debería ser fecha/hora)
zf_pagos.fecha_pago         tipo actual: str  (debería ser fecha/hora)
zf_accesos.fecha_hora         tipo actual: str  (debería ser fecha/hora)
kg_actividad.visit_date         tipo actual: str  (debería ser fecha/hora)
kg_actividad.check_in_time      tipo actual: str  (debería ser fecha/hora)


Las **8 columnas de fecha u hora** se cargaron como texto (`str`). Mientras no se conviertan con `pd.to_datetime()`, no se puede calcular la antigüedad de un miembro, los días entre visitas ni la franja horaria, que son variables de la pregunta analítica.

Ahora se revisa por qué `zf_pagos.valor` quedó como texto: se intenta convertir a número y se muestran los valores que fallan.

In [32]:
valor_numerico = pd.to_numeric(zf_pagos["valor"], errors="coerce")
valores_no_numericos = zf_pagos.loc[valor_numerico.isna(), "valor"]

print("Pagos con valor no numérico:", len(valores_no_numericos))
print(valores_no_numericos.value_counts().head(8))

Pagos con valor no numérico: 179
valor
$90.000     60
$70.000     59
$60.000     14
$240.000    12
$450.000     8
$120.000     8
$420.000     4
$81.000      4
Name: count, dtype: int64


**179 pagos** tienen el monto escrito con formato de moneda (`$90.000`, con signo de pesos y punto de miles). Basta uno de esos valores para que Pandas cargue toda la columna como texto, y así no se pueden sumar ni promediar los pagos.

Se revisa también el formato de las fechas de pago:

In [33]:
formato_iso = zf_pagos["fecha_pago"].str.match(r"^\d{4}-\d{2}-\d{2}$")
print("Fechas AAAA-MM-DD:", formato_iso.sum())
print("Fechas en otro formato:", (~formato_iso).sum())
print(zf_pagos.loc[~formato_iso, "fecha_pago"].head())

Fechas AAAA-MM-DD: 3430
Fechas en otro formato: 142
26    20/10/2025
28    01/06/2026
76    02/02/2026
85    19/12/2025
89    17/05/2025
Name: fecha_pago, dtype: str


**142 fechas de pago** están en formato `DD/MM/AAAA` en lugar de `AAAA-MM-DD`. Si se convierten sin indicar el formato, la conversión falla o confunde el día con el mes.

Otros tipos que no corresponden al significado de la variable:

In [34]:
print("zf_planes.ingresos_incluidos:", zf_planes["ingresos_incluidos"].dtype, "-> es un conteo entero")

meses = kg_churn["Month_to_end_contract"]
print("kg_churn.Month_to_end_contract:", meses.dtype, "| todos enteros:", (meses % 1 == 0).all())

binarias = ["gender", "Near_Location", "Partner", "Promo_friends", "Phone", "Group_visits", "Churn"]
print("kg_churn columnas binarias 0/1 guardadas como int64:", binarias)

zf_planes.ingresos_incluidos: float64 -> es un conteo entero
kg_churn.Month_to_end_contract: float64 | todos enteros: True
kg_churn columnas binarias 0/1 guardadas como int64: ['gender', 'Near_Location', 'Partner', 'Promo_friends', 'Phone', 'Group_visits', 'Churn']


- `ingresos_incluidos` y `Month_to_end_contract` son conteos enteros guardados como decimales (`float64`).
- En `kg_churn`, 7 columnas son categorías (sí/no) guardadas como números 0/1. Sirven para modelar, pero para describirlas hay que tratarlas como categorías y conocer qué significa cada código.

## 4.3 Posibles inconsistencias

### a) Categorías escritas de varias formas (`zf_miembros`)

In [35]:
print(zf_miembros["genero"].value_counts(dropna=False))
print()
print(zf_miembros["comuna"].value_counts(dropna=False).tail(4))

genero
M            446
F            402
NaN           27
f             15
Masculino      7
masculino      6
Femenino       5
Name: count, dtype: int64

comuna
Guayabal           32
Doce de Octubre    26
Aranjuez           25
laureles           18
Name: count, dtype: int64


`genero` debería tener 2 valores (`M` y `F`), pero tiene 6: **33 registros** usan `f`, `Masculino`, `masculino` o `Femenino`. En `comuna`, **18 registros** dicen `laureles` en lugar de `Laureles-Estadio`. Si no se unifican, al agrupar por género o comuna los mismos grupos aparecen divididos.

### b) Fechas de nacimiento de relleno y edades

In [36]:
nacimiento = pd.to_datetime(zf_miembros["fecha_nacimiento"], errors="coerce")
edad = ((FECHA_CORTE - nacimiento).dt.days // 365).astype("Int64")

print("Fechas 1900-01-01:", (zf_miembros["fecha_nacimiento"] == "1900-01-01").sum())
print("Edad máxima calculada:", edad.max())
print("Miembros menores de 18 años:", (edad < 18).sum(), "| edad mínima:", edad.min())

Fechas 1900-01-01: 9


Edad máxima calculada: 126
Miembros menores de 18 años: 84 | edad mínima: 15


**9 miembros** tienen fecha de nacimiento `1900-01-01`, lo que da una edad de 126 años: es un valor de relleno que se usó cuando no se conocía el dato. Esas 9 fechas deben tratarse como faltantes. Además, **84 miembros** tienen entre 15 y 17 años. No es un error en sí, pero el gimnasio debe confirmar si admite menores y con qué autorización.

### c) Miembros duplicados

In [37]:
doc_repetido = zf_miembros["documento_hash"].duplicated(keep=False)
print("Registros con documento repetido:", doc_repetido.sum())
display(zf_miembros[doc_repetido].sort_values("documento_hash").head(6))

sin_membresia = ~zf_miembros["id_miembro"].isin(zf_membresias["id_miembro"])
print("Miembros sin ninguna membresía:", sin_membresia.sum())
print("IDs:", zf_miembros.loc[sin_membresia, "id_miembro"].tolist())

Registros con documento repetido: 16


,id_miembro,documento_hash,genero,fecha_nacimiento,comuna,canal_captacion,fecha_registro
841,842,DOC14355577,M,1996-03-19,Castilla,Convenio empresarial,2026-06-08
905,906,DOC14355577,M,1996-03-19,Castilla,Convenio empresarial,2026-06-08
256,257,DOC16318631,M,1990-04-28,La América,Volante,2026-03-29
900,901,DOC16318631,M,1990-04-28,La América,Volante,2026-03-29
259,260,DOC16319341,F,1997-05-06,Belén,Convenio empresarial,2025-11-22
902,903,DOC16319341,F,1997-05-06,Belén,Convenio empresarial,2025-11-22


Miembros sin ninguna membresía: 8
IDs: [901, 902, 903, 904, 905, 906, 907, 908]


**8 personas están inscritas dos veces** (16 registros con el mismo documento y los mismos datos). Las copias son los `id_miembro` 901 a 908, y son justamente los **8 miembros sin ninguna membresía**. Si no se eliminan, inflan el total de miembros y aparecen como personas que nunca compraron un plan.

### d) Estado de las membresías frente a sus fechas

In [38]:
fecha_fin = pd.to_datetime(zf_membresias["fecha_fin"])
activa_vencida = (zf_membresias["estado"] == "Activa") & (fecha_fin < FECHA_CORTE)
print("Membresías 'Activa' con fecha_fin anterior al corte:", activa_vencida.sum())

activas_por_miembro = zf_membresias[zf_membresias["estado"] == "Activa"].groupby("id_miembro").size()
print("Miembros con más de una membresía 'Activa':", (activas_por_miembro > 1).sum())

Membresías 'Activa' con fecha_fin anterior al corte: 93
Miembros con más de una membresía 'Activa': 41


**93 membresías siguen marcadas como `Activa` aunque su fecha de fin ya pasó**. La regla del negocio dice que deben pasar a `Vencida` automáticamente. Por eso **41 miembros** aparecen con dos membresías activas al mismo tiempo. Es el hallazgo más crítico: la deserción se mide con el estado y las fechas de la membresía, y con estos estados se contaría como activo a quien ya se fue.

### e) Pagos duplicados, en cero o sin método

In [39]:
print("id_pago repetidos:", zf_pagos["id_pago"].duplicated().sum())
print("Filas completamente duplicadas:", zf_pagos.duplicated().sum())

pago_cero = (valor_numerico == 0) & (zf_pagos["estado_pago"] == "Aprobado")
print("Pagos aprobados con valor 0:", pago_cero.sum())
display(zf_pagos[pago_cero].head(5))

id_pago repetidos: 12
Filas completamente duplicadas: 12
Pagos aprobados con valor 0: 14


,id_pago,id_membresia,fecha_pago,valor,metodo_pago,estado_pago
124,125,116,2026-01-12,0,Nequi,Aprobado
705,706,671,2025-08-21,0,Efectivo,Aprobado
773,774,734,2026-01-15,0,Transferencia,Aprobado
1039,1040,994,2025-09-15,0,Nequi,Aprobado
1055,1056,1006,2026-05-27,0,Tarjeta,Aprobado


- **12 pagos están duplicados**: se repite la fila completa, incluido el `id_pago`. Si se suman, los ingresos quedan inflados.
- **14 pagos aprobados tienen valor 0**: una membresía se habría pagado con cero pesos, lo que no es válido.

### f) Accesos: integridad, horario y lecturas dobles

In [40]:
miembro_inexistente = ~zf_accesos["id_miembro"].isin(zf_miembros["id_miembro"])
print("Accesos de miembros que no existen:", miembro_inexistente.sum())
print("IDs distintos:", zf_accesos.loc[miembro_inexistente, "id_miembro"].nunique())

fecha_hora = pd.to_datetime(zf_accesos["fecha_hora"])
fuera_horario = (fecha_hora.dt.hour < 5) | (fecha_hora.dt.hour >= 22)
print("Accesos fuera del horario 5:00-22:00:", fuera_horario.sum())
print(zf_accesos.loc[fuera_horario, "resultado"].value_counts())

Accesos de miembros que no existen: 25
IDs distintos: 20


Accesos fuera del horario 5:00-22:00: 17
resultado
Permitido    17
Name: count, dtype: int64


In [41]:
accesos_ordenados = zf_accesos.assign(fecha_hora_dt=fecha_hora).sort_values(["id_miembro", "fecha_hora_dt"])
segundos = accesos_ordenados.groupby("id_miembro")["fecha_hora_dt"].diff().dt.total_seconds()
lectura_doble = segundos < 60

print("Accesos a menos de 60 s del anterior del mismo miembro:", lectura_doble.sum())
print(segundos[lectura_doble].describe().round(1))

Accesos a menos de 60 s del anterior del mismo miembro: 675
count    675.0
mean      26.8
std       13.0
min        5.0
25%       15.0
50%       27.0
75%       38.0
max       49.0
Name: fecha_hora_dt, dtype: float64


- **25 accesos** pertenecen a **20 miembros que no existen** en `zf_miembros` (IDs del 953 al 989): el lector reconoce huellas que no están asociadas a ningún miembro registrado.
- **17 accesos** ocurrieron fuera del horario de la sede (entre las 0:00 y las 3:00 y a las 22:00) y todos fueron `Permitido`.
- **675 accesos** se registraron entre 5 y 49 segundos después del acceso anterior del mismo miembro: **el lector leyó dos veces la misma huella**. Esto infla la frecuencia de asistencia, que es una variable de la pregunta analítica.

### g) Accesos permitidos sin membresía vigente y tiquetera por encima del límite

In [42]:
membresias_fechas = zf_membresias.assign(
    inicio=pd.to_datetime(zf_membresias["fecha_inicio"]),
    fin=pd.to_datetime(zf_membresias["fecha_fin"]),
)
accesos_dia = zf_accesos.assign(dia=fecha_hora.dt.normalize())

cruce = accesos_dia.merge(
    membresias_fechas[["id_miembro", "id_membresia", "id_plan", "inicio", "fin", "estado"]],
    on="id_miembro",
)
cruce["vigente"] = (cruce["dia"] >= cruce["inicio"]) & (cruce["dia"] <= cruce["fin"])

vigente_no_cancelada = cruce[cruce["vigente"] & (cruce["estado"] != "Cancelada")]
con_cancelada = cruce[cruce["vigente"] & (cruce["estado"] == "Cancelada")]

ids_validos = set(vigente_no_cancelada["id_acceso"])
permitidos = accesos_dia[accesos_dia["resultado"] == "Permitido"]
sin_vigencia = permitidos[~permitidos["id_acceso"].isin(ids_validos) & permitidos["id_miembro"].isin(zf_miembros["id_miembro"])]

print("Accesos 'Permitido' de miembros existentes sin membresía vigente:", len(sin_vigencia))
print("De ellos, durante una membresía 'Cancelada':", sin_vigencia["id_acceso"].isin(con_cancelada["id_acceso"]).sum())

Accesos 'Permitido' de miembros existentes sin membresía vigente: 184
De ellos, durante una membresía 'Cancelada': 184


In [43]:
TIQUETERA = 5  # id_plan de la tiquetera de 12 ingresos

uso_tiquetera = vigente_no_cancelada[
    (vigente_no_cancelada["id_plan"] == TIQUETERA) & (vigente_no_cancelada["resultado"] == "Permitido")
]
ingresos = uso_tiquetera.groupby("id_membresia").size()
print("Tiqueteras con más de 12 ingresos:", (ingresos > 12).sum(), "| máximo:", ingresos.max())

ids_dobles = set(accesos_ordenados.loc[lectura_doble, "id_acceso"])
ingresos_sin_dobles = uso_tiquetera[~uso_tiquetera["id_acceso"].isin(ids_dobles)].groupby("id_membresia").size()
print("Sin contar lecturas dobles:", (ingresos_sin_dobles > 12).sum(), "| máximo:", ingresos_sin_dobles.max())

Tiqueteras con más de 12 ingresos: 22 | máximo: 14


Sin contar lecturas dobles: 1 | máximo: 13


- **184 accesos fueron `Permitido` durante una membresía `Cancelada`.** La regla dice que la puerta solo abre con membresía activa, así que el control de acceso no está respetando las cancelaciones.
- **22 tiqueteras** muestran más de 12 ingresos, pero al quitar las lecturas dobles **solo queda 1** (13 ingresos). La mayoría del exceso lo causa el lector, no los miembros. Esto confirma que primero hay que eliminar las lecturas dobles y después medir la asistencia.

### h) Datasets externos

In [44]:
ausente = kg_actividad["attendance_status"] == "Absent"
print("kg_actividad – visitas 'Absent':", ausente.sum())
print("  con duración > 0:", (kg_actividad.loc[ausente, "workout_duration_minutes"] > 0).sum())
print("  con calorías > 0:", (kg_actividad.loc[ausente, "calories_burned"] > 0).sum())
print("  rango de fechas:", kg_actividad["visit_date"].min(), "a", kg_actividad["visit_date"].max())

kg_actividad – visitas 'Absent': 1350
  con duración > 0: 1350
  con calorías > 0: 1350
  rango de fechas: 2024-01-01 a 2024-12-31


In [45]:
fuera_medellin = ~gov_escenarios["latitud"].between(6.1, 6.4) | ~gov_escenarios["longitud"].between(-75.75, -75.45)
print("gov_escenarios – coordenadas fuera de Medellín:", fuera_medellin.sum())
display(gov_escenarios.loc[fuera_medellin, ["nombre_escenario", "barrio", "latitud", "longitud"]].head(6))

espacios = gov_escenarios["direccion"].str.contains(r"\s{2,}")
print("Direcciones con espacios múltiples:", espacios.sum())

gov_escenarios – coordenadas fuera de Medellín: 22


,nombre_escenario,barrio,latitud,longitud
6,Cancha de baloncesto Campo Amor,Campo Amor,55.555555,55.555555
39,Cancha de baloncesto Parque Las Brisas,Las Brisas,3.703704,3.703704
114,Cancha de fútbol en grama sintética Desarrollo Deportivo Integral El Salado,El Salado,8.725338,-73.158492
153,Cancha de fútbol en grama sintética UVA Sol de Oriente,Trece de Noviembre,1.234568,1.234568
178,Cancha de microfútbol en grama sintética N 1 Unidad Deportiva Juanes de la Paz,Girardot,7.529940,-74.333540
179,Cancha de microfútbol en grama sintética N 2 Unidad Deportiva Juanes de la Paz,Girardot,7.529940,-74.333540


Direcciones con espacios múltiples: 966


- **`kg_actividad`**: las **1.350 visitas `Absent`** tienen duración de entrenamiento y calorías quemadas mayores que cero. Una persona ausente no puede haber entrenado 20 minutos o más: la variable `attendance_status` no es confiable.
- **`gov_escenarios`**: **22 escenarios** tienen coordenadas fuera de Medellín, con valores de relleno como `1.234568` o `55.555555`. Si se ubican en un mapa, aparecerían en el mar, en otros países o en otras regiones de Colombia. Además, **966 direcciones** tienen espacios múltiples.
- **`kg_churn`** no presenta inconsistencias internas, pero sus códigos 0/1 no están documentados.

## 4.4 Limitaciones de los datos

1. **Datos simulados.** Las tablas de Zona Fit imitan la base de datos real, pero no son observaciones reales. Los resultados sirven para construir y probar el proceso de análisis; las conclusiones de negocio deben validarse cuando el gimnasio tenga datos de operación.
2. **Solo se registra la entrada.** `zf_accesos` no tiene hora de salida, así que no se puede medir cuánto tiempo entrena cada miembro, solo cuándo llega.
3. **No hay motivo de retiro.** Las membresías dicen si se vencieron o cancelaron, pero no por qué. La pregunta analítica busca factores asociados, no causas declaradas.
4. **Periodo corto.** Los datos van de enero de 2025 a octubre de 2026 (unos 21 meses). Los miembros inscritos en los últimos seis meses todavía no completan su periodo de observación para la pregunta.
5. **Datasets externos de otro contexto.** `kg_churn` y `kg_actividad` no son de Colombia, no tienen fechas ni identificadores en común con Zona Fit y no se pueden unir con sus tablas. Sirven como referencia para comparar variables, no para combinarse fila a fila.
6. **Granularidad geográfica distinta.** `gov_escenarios` está por barrio y `zf_miembros` por comuna. Para relacionarlos hace falta una tabla de equivalencias barrio–comuna o usar las coordenadas.
7. **`kg_actividad` tiene una visita por miembro.** No permite estudiar la asistencia de una persona a lo largo del tiempo.

## 4.5 Tabla de hallazgos y acciones propuestas

La columna *Evidencia encontrada* usa las variables calculadas arriba, para que coincida con los resultados de Pandas.

In [46]:
hallazgos = [
    ("zf_miembros", "Valores faltantes en género, nacimiento y comuna",
     f"isna(): genero {zf_miembros['genero'].isna().sum()}, fecha_nacimiento {zf_miembros['fecha_nacimiento'].isna().sum()}, comuna {zf_miembros['comuna'].isna().sum()}",
     "Perfiles incompletos; no se calcula la edad de esos miembros",
     "Marcar como 'Sin dato'; no imputar edad; pedir el dato al renovar"),
    ("zf_miembros", "Categorías inconsistentes en genero y comuna",
     f"genero con {zf_miembros['genero'].nunique()} valores; comuna 'laureles' {(zf_miembros['comuna'] == 'laureles').sum()} veces",
     "Grupos divididos al agrupar por género o comuna",
     "Unificar con un diccionario de reemplazo (map/replace)"),
    ("zf_miembros", "Fecha de nacimiento de relleno",
     f"'1900-01-01' en {(zf_miembros['fecha_nacimiento'] == '1900-01-01').sum()} registros (edad {int(edad.max())} años)",
     "Edades imposibles que distorsionan promedios",
     "Convertir 1900-01-01 en NaN; validar rango de edad en el formulario"),
    ("zf_miembros", "Miembros inscritos dos veces",
     f"{doc_repetido.sum()} registros con documento repetido; {sin_membresia.sum()} sin membresía (IDs 901-908)",
     "Infla el total de miembros",
     "Eliminar duplicados por documento_hash; restricción UNIQUE en MySQL"),
    ("zf_membresias", "Estado 'Activa' con fecha vencida",
     f"{activa_vencida.sum()} membresías; {(activas_por_miembro > 1).sum()} miembros con más de una activa",
     "Subestima la deserción (variable central de la pregunta)",
     "Recalcular el estado con fecha_fin y la fecha de corte; tarea automática en el backend"),
    ("zf_membresias", "Precio pagado faltante",
     f"isna(): precio_pagado {zf_membresias['precio_pagado'].isna().sum()}",
     "Ingresos subestimados",
     "Completar con la suma de pagos aprobados o el precio de lista del plan"),
    ("zf_pagos", "Monto guardado como texto con formato de moneda",
     f"dtype {zf_pagos['valor'].dtype}; {len(valores_no_numericos)} valores tipo '$90.000'",
     "No se puede sumar ni promediar",
     "Quitar '$' y '.', convertir con pd.to_numeric"),
    ("zf_pagos", "Fechas en dos formatos",
     f"{(~formato_iso).sum()} fechas DD/MM/AAAA",
     "Conversión fallida o día y mes invertidos",
     "Convertir cada formato con su format explícito en pd.to_datetime"),
    ("zf_pagos", "Pagos duplicados",
     f"{zf_pagos.duplicated().sum()} filas idénticas (id_pago repetido)",
     "Ingresos inflados",
     "drop_duplicates(); id_pago como PRIMARY KEY"),
    ("zf_pagos", "Pagos aprobados en cero y método vacío",
     f"{pago_cero.sum()} pagos con valor 0; metodo_pago vacío {zf_pagos['metodo_pago'].isna().sum()}",
     "Membresías aparentemente gratis; análisis por medio de pago incompleto",
     "Revisar los pagos en cero con caja; método 'Sin dato'"),
    ("zf_accesos", "Fechas como texto y dispositivo vacío",
     f"fecha_hora dtype {zf_accesos['fecha_hora'].dtype}; dispositivo vacío {zf_accesos['dispositivo'].isna().sum()}",
     "No se calcula franja horaria ni frecuencia sin convertir",
     "pd.to_datetime; imputar 'LECTOR-01' (único lector)"),
    ("zf_accesos", "Lecturas dobles de huella",
     f"{lectura_doble.sum()} accesos a menos de 60 s del anterior",
     "Sobreestima la frecuencia de asistencia",
     "Eliminar accesos a menos de 60 s del anterior del mismo miembro"),
    ("zf_accesos", "Accesos de miembros inexistentes",
     f"{miembro_inexistente.sum()} accesos de {zf_accesos.loc[miembro_inexistente, 'id_miembro'].nunique()} IDs no registrados",
     "Rompe la integridad entre tablas",
     "Excluir del análisis; llave foránea en MySQL"),
    ("zf_accesos", "Accesos permitidos con membresía cancelada",
     f"{len(sin_vigencia)} accesos 'Permitido' sin membresía vigente",
     "Asistencia atribuida a quien ya no es miembro",
     "Excluir del análisis; corregir la validación del lector"),
    ("zf_accesos", "Accesos fuera del horario",
     f"{fuera_horario.sum()} accesos entre 22:00 y 5:00",
     "Registros dudosos en el análisis de franjas",
     "Revisar con el administrador; excluir si no son válidos"),
    ("kg_churn", "Variables codificadas sin diccionario",
     "7 columnas binarias 0/1; Month_to_end_contract float64",
     "Interpretación incierta de gender",
     "Documentar códigos; convertir a categoría y a entero"),
    ("kg_actividad", "Ausencias con entrenamiento registrado",
     f"{(kg_actividad.loc[ausente, 'workout_duration_minutes'] > 0).sum()} visitas 'Absent' con duración y calorías > 0",
     "attendance_status no es confiable",
     "No usar attendance_status; usar solo como referencia de horarios"),
    ("gov_escenarios", "Coordenadas fuera de Medellín",
     f"{fuera_medellin.sum()} escenarios con latitud/longitud imposibles",
     "Ubicaciones erradas en mapas",
     "Marcar como NaN y geocodificar desde la dirección"),
    ("gov_escenarios", "Direcciones con espacios y sin comuna",
     f"{espacios.sum()} direcciones con espacios múltiples; sin columna comuna",
     "No se cruza con zf_miembros",
     "Normalizar espacios; agregar comuna con tabla barrio-comuna"),
]

tabla_hallazgos = pd.DataFrame(
    hallazgos,
    columns=["Dataset", "Problema identificado", "Evidencia encontrada", "Posible impacto", "Acción propuesta"],
)
print("Total de hallazgos:", len(tabla_hallazgos))
display(tabla_hallazgos)

Total de hallazgos: 19


,Dataset,Problema identificado,Evidencia encontrada,Posible impacto,Acción propuesta
0,zf_miembros,"Valores faltantes en género, nacimiento y comuna","isna(): genero 27, fecha_nacimiento 54, comuna 46",Perfiles incompletos; no se calcula la edad de esos miembros,Marcar como 'Sin dato'; no imputar edad; pedir el dato al renovar
1,zf_miembros,Categorías inconsistentes en genero y comuna,genero con 6 valores; comuna 'laureles' 18 veces,Grupos divididos al agrupar por género o comuna,Unificar con un diccionario de reemplazo (map/replace)
2,zf_miembros,Fecha de nacimiento de relleno,'1900-01-01' en 9 registros (edad 126 años),Edades imposibles que distorsionan promedios,Convertir 1900-01-01 en NaN; validar rango de edad en el formulario
3,zf_miembros,Miembros inscritos dos veces,16 registros con documento repetido; 8 sin membresía (IDs 901-908),Infla el total de miembros,Eliminar duplicados por documento_hash; restricción UNIQUE en MySQL
4,zf_membresias,Estado 'Activa' con fecha vencida,93 membresías; 41 miembros con más de una activa,Subestima la deserción (variable central de la pregunta),Recalcular el estado con fecha_fin y la fecha de corte; tarea automática en el backend
5,zf_membresias,Precio pagado faltante,isna(): precio_pagado 51,Ingresos subestimados,Completar con la suma de pagos aprobados o el precio de lista del plan
6,zf_pagos,Monto guardado como texto con formato de moneda,dtype str; 179 valores tipo '$90.000',No se puede sumar ni promediar,"Quitar '$' y '.', convertir con pd.to_numeric"
7,zf_pagos,Fechas en dos formatos,142 fechas DD/MM/AAAA,Conversión fallida o día y mes invertidos,Convertir cada formato con su format explícito en pd.to_datetime
8,zf_pagos,Pagos duplicados,12 filas idénticas (id_pago repetido),Ingresos inflados,drop_duplicates(); id_pago como PRIMARY KEY
9,zf_pagos,Pagos aprobados en cero y método vacío,14 pagos con valor 0; metodo_pago vacío 108,Membresías aparentemente gratis; análisis por medio de pago incompleto,Revisar los pagos en cero con caja; método 'Sin dato'


## Conclusiones

1. **Los datos sirven para continuar con el proyecto, pero no se pueden analizar todavía en su estado actual.** Los faltantes son bajos (ninguno supera el 6 % en columnas relevantes); los problemas más graves son de **consistencia**, no de completitud.
2. **El hallazgo más crítico** son las **93 membresías que siguen `Activa` aunque ya vencieron**, porque afectan directamente la variable de deserción, que es el centro de la pregunta analítica.
3. **El segundo** son las **675 lecturas dobles de huella**, que inflan la frecuencia de asistencia: explican casi todo el exceso de ingresos en las tiqueteras.
4. **Antes del análisis** hay que: recalcular estados de membresía, convertir fechas y montos a su tipo correcto, unificar categorías, eliminar duplicados (miembros, pagos y lecturas dobles) y excluir accesos inválidos.
5. **Los datasets externos** se usarán solo como referencia: `kg_churn` para comparar variables asociadas a la deserción, `kg_actividad` para patrones de horario (sin usar `attendance_status`) y `gov_escenarios` como contexto geográfico, después de corregir sus coordenadas.
6. Varios hallazgos también son **mejoras para el sistema Zona Fit Evolution**: restricción `UNIQUE` en el documento, llaves foráneas, actualización automática del estado de membresía y validación del lector de huella.